# IN6227 Assignment 1 — Variant 1
## Part 3: Feature Engineering

Models chosen: **Logistic Regression (LR)** vs **Random Forest (RF)**.

In [1]:
import time
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

pd.set_option("display.width", 200)
RANDOM_STATE = 42
TARGET = "label"

train = pd.read_csv("train_clean.csv")
X_train = train.drop(columns=TARGET)
y_train = train[TARGET]

num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]
print("X_train:", X_train.shape, "| positive rate:", round(y_train.mean(), 3))
print("Numeric:", num_cols)
print("Categorical:", cat_cols)

X_train: (31109, 15) | positive rate: 0.24
Numeric: ['aptitude_score', 'performance_score', 'stability_index', 'load_ratio', 'index_weight', 'activity_duration', 'composite_rank']
Categorical: ['geological_era', 'weather_pattern', 'region', 'personal_interest', 'brew_preference', 'instrument', 'species', 'mineral_type']


### 3.1 The preprocessing building block

In [3]:
def make_preprocessor(scale, drop=()):
    num = [c for c in num_cols if c not in drop]
    cat = [c for c in cat_cols if c not in drop]

    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))

    cat_steps = [("impute", SimpleImputer(strategy="most_frequent")),
                 ("onehot", OneHotEncoder(handle_unknown="ignore"))]

    return ColumnTransformer([
        ("num", Pipeline(num_steps), num),
        ("cat", Pipeline(cat_steps), cat),
    ])

# Quick look: what does the preprocessed data look like?
pre = make_preprocessor(scale=True).fit(X_train)
Xt = pre.transform(X_train)
print("Before preprocessing:", X_train.shape, "-> after:", Xt.shape)
print("First 10 output columns:", list(pre.get_feature_names_out()[:10]))

Before preprocessing: (31109, 15) -> after: (31109, 63)
First 10 output columns: ['num__aptitude_score', 'num__performance_score', 'num__stability_index', 'num__load_ratio', 'num__index_weight', 'num__activity_duration', 'num__composite_rank', 'cat__geological_era_Devonian', 'cat__geological_era_Jurassic', 'cat__geological_era_Other']


In [4]:
# Check that scaling did what we expect: numeric columns now have mean ~0 and std ~1
Xt_dense = Xt.toarray() if hasattr(Xt, "toarray") else Xt
print(pd.DataFrame({
    "mean": Xt_dense[:, :len(num_cols)].mean(axis=0).round(3),
    "std": Xt_dense[:, :len(num_cols)].std(axis=0).round(3),
}, index=num_cols))

                   mean  std
aptitude_score      0.0  1.0
performance_score   0.0  1.0
stability_index    -0.0  1.0
load_ratio         -0.0  1.0
index_weight       -0.0  1.0
activity_duration  -0.0  1.0
composite_rank     -0.0  1.0


### 3.2 Evidence of redundancy: is `composite_rank` derived from other columns?

In [6]:
d = X_train[num_cols].dropna()
for cols in [["performance_score"], ["aptitude_score", "performance_score"]]:
    reg = LinearRegression().fit(d[cols], d["composite_rank"])
    print(f"composite_rank ~ {cols}: R2 = {reg.score(d[cols], d['composite_rank']):.6f}",
          "| coefficients:", np.round(reg.coef_, 5))

composite_rank ~ ['performance_score']: R2 = 0.774182 | coefficients: [0.00408]
composite_rank ~ ['aptitude_score', 'performance_score']: R2 = 1.000000 | coefficients: [0.006 0.004]


In [8]:
Z = X_train[num_cols].fillna(X_train[num_cols].median())
vif = {}
for c in num_cols:
    others = [o for o in num_cols if o != c]
    r2 = LinearRegression().fit(Z[others], Z[c]).score(Z[others], Z[c])
    vif[c] = 1 / (1 - r2)
print(pd.Series(vif).round(1).sort_values(ascending=False))

composite_rank       8535.1
performance_score    6356.4
aptitude_score       1932.4
load_ratio              9.3
activity_duration       7.0
stability_index         5.8
index_weight            1.0
dtype: float64


### 3.3 Experiment: which feature set works best for Logistic Regression?

In [10]:
REDUNDANT_DERIVED = ["composite_rank"]
REDUNDANT_PAIR = ["performance_score", "stability_index"]
LOW_INFO = ["index_weight", "region", "geological_era"]

feature_sets = {
    "A. all features":                        dict(scale=True,  drop=[]),
    "B. all features, NO scaling":            dict(scale=False, drop=[]),
    "C. drop composite_rank":                 dict(scale=True,  drop=REDUNDANT_DERIVED),
    "D. drop performance + stability":        dict(scale=True,  drop=REDUNDANT_PAIR),
    "E. drop low-information features":       dict(scale=True,  drop=LOW_INFO),
    "F. drop composite_rank + low-info":      dict(scale=True,  drop=REDUNDANT_DERIVED + LOW_INFO),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, cfg in feature_sets.items():
    pipe = Pipeline([("pre", make_preprocessor(**cfg)),
                     ("model", LogisticRegression(max_iter=1000))])
    t0 = time.time()
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=["f1", "roc_auc"])
    rows.append([name, len(num_cols) + len(cat_cols) - len(cfg["drop"]),
                 res["test_f1"].mean(), res["test_f1"].std(),
                 res["test_roc_auc"].mean(), res["test_roc_auc"].std(), time.time() - t0])

results = pd.DataFrame(rows, columns=["feature set", "n_features", "F1 mean", "F1 std",
                                      "AUC mean", "AUC std", "time (s)"]).round(4)
results

,feature set,n_features,F1 mean,F1 std,AUC mean,AUC std,time (s)
0,A. all features,15,0.6244,0.0128,0.8906,0.0036,2.4034
1,"B. all features, NO scaling",15,0.6205,0.0134,0.8886,0.0042,15.9839
2,C. drop composite_rank,14,0.6241,0.0127,0.8906,0.0035,2.2136
3,D. drop performance + stability,13,0.6122,0.0109,0.8859,0.0047,2.3746
4,E. drop low-information features,12,0.6233,0.0117,0.8903,0.0036,2.0534
5,F. drop composite_rank + low-info,11,0.6232,0.0121,0.8903,0.0036,1.8555


### 3.4 Decision and final pipelines

In [12]:
LR_DROP = REDUNDANT_DERIVED + LOW_INFO     # decided in 3.3
RF_DROP = []

lr_preprocessor = make_preprocessor(scale=True, drop=LR_DROP)
rf_preprocessor = make_preprocessor(scale=False, drop=RF_DROP)

for name, p, drop in [("LR", lr_preprocessor, LR_DROP), ("RF", rf_preprocessor, RF_DROP)]:
    p.fit(X_train)
    print(f"{name}: {X_train.shape[1] - len(drop)} input features "
          f"-> {len(p.get_feature_names_out())} columns after one-hot encoding")
print("\nLR drops:", LR_DROP)

LR: 11 input features -> 53 columns after one-hot encoding
RF: 15 input features -> 63 columns after one-hot encoding

LR drops: ['composite_rank', 'index_weight', 'region', 'geological_era']
